## <font color='red'> INSTRUCTIONS </font>

<b> 
1. Write your code only in cells below the "WRITE CODE BELOW" title. Do not modify the code below the "DO NOT MODIFY" title. <br>
2. The expected data types of the output answers for each question are given in the last cell through assertion statements. Your answers must match these expected output data types. Hint: Many of the answers need to be a Python dictionary. Consider methods like to_dict() to convert a Pandas Series to a dictionary. <br>
3. The answers are then written to a JSON file named my_results_PA1.json. You can compare this with the provided expected output file "expected_results_PA1.json". <br>
4. After you complete writing your code, click "Kernel -> Restart Kernel and Run All Cells" on the top toolbar. There should NOT be any syntax/runtime errors, otherwise points will be deducted. <br>
5. For submitting your solution, first download your notebook by clicking "File -> Download". Rename the file as &ltTEAM_ID&gt.ipynb" and upload to Canvas.</b>


## <font color='red'> DO NOT MODIFY </font>

In [1]:
import time
import json
import dask
import dask.dataframe as dd
import pandas as pd
import ast
import re
from dask.distributed import Client
import ctypes
import numpy as np

import ast
import itertools

def trim_memory() -> int:
    """
    helps to fix any memory leaks.
    """
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client = Client("127.0.0.1:8786")
client.run(trim_memory)
client = client.restart()
print(client)

None


In [2]:
start = time.time()

## <font color='blue'> WRITE CODE BELOW </font>

In [3]:
user_reviews = dd.read_csv('user_reviews.csv')
products = dd.read_csv('products.csv', dtype={'asin': 'object'})

num_reviews = len(user_reviews)
num_products = len(products)

In [4]:
## Question 1
# Time: 30 seconds
null_percent = ((user_reviews.isna().sum() / num_reviews) * 100).round(2).compute()
ans1 = null_percent.to_dict()
ans1

{'reviewerID': 0.0,
 'asin': 0.0,
 'reviewerName': 3.29,
 'helpful': 0.0,
 'reviewText': 0.02,
 'overall': 0.0,
 'summary': 0.01,
 'unixReviewTime': 0.0,
 'reviewTime': 0.0}

In [5]:
## Question 2
# Time: 18.55 seconds
null_percent = ((products.isna().sum() / num_products) * 100).round(2).compute()
ans2 = null_percent.to_dict()
ans2

{'asin': 0.0,
 'salesRank': 26.02,
 'imUrl': 1.8,
 'categories': 0.8,
 'title': 15.19,
 'description': 41.72,
 'price': 35.7,
 'related': 29.04,
 'brand': 81.69}

In [10]:
# ## Question 3
# Time: 56.6 seconds
# Needed to speed this up, trying to index by asin
#user_reviews_indexed = user_reviews.set_index('asin')
#products_indexed = products.set_index('asin')
#product_price = products_indexed[['price']]

# Need to join so that all user reviews have the price of the associated product
#joined = user_reviews_indexed.merge(product_price, how='left',indicator=True)
start3 = time.time()

joined = user_reviews[['asin', 'overall']].merge(products[['asin', 'price']], on = 'asin', how = 'left', indicator = True)

# This is a 2x2 dataframe
corr = joined[['price', 'overall']].corr(method='pearson').compute()
ans3 = float(corr['price'].iloc[1].round(2))
end3 = time.time()
print(start3-end3)
ans3

-52.29804468154907


-0.01

In [7]:
## Question 4
# Time: 11.4 seconds
desc = products['price'].describe().compute()
ans4 = {'mean': desc['mean'], 'std': desc['std'], 'min': desc['min'], 'max': desc['max'],'median': desc['50%']}
ans4

{'mean': np.float64(34.937356116908504),
 'std': np.float64(71.26369249877509),
 'min': np.float64(0.0),
 'max': np.float64(999.99),
 'median': np.float64(19.55)}

In [8]:
## Question 5
# Time: 13.67 seconds

# Idea: Process the category column from the products table and get first value from each list
# Groupby this processed column and use .count()
# Sort in non-increasing order
   
first_cat = products['categories'].dropna().str.extract(r"\[\['([^']+)")[0]
products = products.assign(first_cat=first_cat)
selected = products[['asin', 'first_cat']].groupby(by='first_cat').count().sort_values(by='asin',ascending=False).compute()
ans5 = selected.to_dict()

In [18]:
client = Client('tcp://172.31.42.95:8786')
from distributed import Client, as_completed


In [22]:
## Question 6
# time is 1 min
#joined = user_reviews_indexed.merge(product_price,how='left',indicator=True)
#ans6 = 1 if len(joined[joined['_merge']=='left_only']) > 0 else 0
#ans6

def is_left_only(df):
    for i in df['_merge']:
        if i == 'left_only':
            return 1
    return 0

partition = joined.map_partitions(lambda x : is_left_only(x)).to_delayed()

# Start every delayed and get the corresponding futures
future_results = client.compute(partition)

# Loop on every partition result as soon as they arrive, and just break when True
for future in as_completed(future_results):
    res = future.result()[0]
    if res:
       ans6 = 1
       break
ans6

1


1

In [ ]:
"""partition_calls = user_reviews.map_partitions(my_func).to_delayed()

# Start every delayed and get the corresponding futures
future_results = client.compute(partition_calls)

# Loop on every partition result as soon as they arrive, and just break when True
for future in as_completed(future_results):
   res = future.result()
   if res:
       break"""

In [ ]:
#products[['asin']].dropna().map_partitions(lambda x : x.isin(reference_set))
"""def in_products(df):
    return set(df['asin']).issubset(reference_set)
user_reviews[['asin']].partitions[0].dropna().map_partitions(lambda x : in_products(x)).compute()"""

In [ ]:
def check_references(lst, valid_set):
    for val in lst:
        if val not in valid_set:
            #print(val)
            return 1
    return 0

reference_set = set(products['asin'].compute())
dangling = 0
for r in range(num_products):
    current = products.loc[r]['related'].dropna().compute()
    # current = products['related'].dropna().compute()
    current = current.astype(str)
    vals = [re.findall(r"'([^,:-]+)'[^:]", c) for c in current]
    #vals = ast.literal_eval
    flattened_list = list(itertools.chain.from_iterable(vals))
    if check_references(flattened_list, reference_set):
        dangling = 1
        break

ans7 = dangling
ans7

In [ ]:
#ast.literal_eval()
#products.loc[1][['related']].dropna().astype(str).map(lambda x : ast.literal_eval(x).values()).compute()

In [ ]:
#len(set(products['asin'].compute()))

In [ ]:
### read in the 'user_reviews.csv' and 'products.csv' files, perform your calculations and place the answers in variables ans1 - ans7.


# substitute 'None' with the outputs from your calculations. 
# The expected output types can be seen in the assertion statements below
ans1 = ans1
ans2 = ans2
ans3 = ans3
ans4 = ans4
ans5 = ans5
ans6 = ans6
ans7 = ans7

## <font color='red'> DO NOT MODIFY </font>

In [ ]:
end = time.time()

In [ ]:
print(f"execution time = {end-start}s")

In [ ]:
# DO NOT MODIFY
assert type(ans1) == dict, f"answer to question 1 must be a dictionary like {{'reviewerID':0.2, ..}}, got type = {type(ans1)}"
assert type(ans2) == dict, f"answer to question 2 must be a dictionary like {{'asin':0.2, ..}}, got type = {type(ans2)}"
assert type(ans3) == float, f"answer to question 3 must be a float like 0.8, got type = {type(ans3)}"
assert type(ans4) == dict, f"answer to question 4 must be a dictionary like {{'mean':0.4,'max':0.6,'median':0.6...}}, got type = {type(ans4)}"
assert type(ans5) == dict, f"answer to question 5 must be a dictionary, got type = {type(ans5)}"         
assert ans6 == 0 or ans6==1, f"answer to question 6 must be 0 or 1, got value = {ans6}" 
assert ans7 == 0 or ans7==1, f"answer to question 7 must be 0 or 1, got value = {ans7}" 

ans_dict = {
    "q1": ans1,
    "q2": ans2,
    "q3": ans3,
    "q4": ans4,
    "q5": ans5,
    "q6": ans6,
    "q7": ans7,
    "runtime": end-start
}
with open('my_results_PA1.json', 'w') as outfile: json.dump(ans_dict, outfile)         